In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import time

import anthropic
from dotenv import load_dotenv

In [3]:
client = anthropic.Anthropic()  # récupère la clé depuis l'environnement, jamais en dur
MODELE = "claude-sonnet-5"       # modèle de réponse prévu par la spec
MODELE_JUGE = "claude-haiku-4-5-20251001"  # futur juge LLM (éval)

Premier appel et anatomie de la réponse

In [4]:
brute = client.messages.with_raw_response.create(
    model=MODELE,
    max_tokens=20,
    messages=[{"role": "user", "content": "Réponds juste : ok"}],
)
print("Workspace :", brute.headers.get("anthropic-workspace-id"))
print("Request ID:", brute.headers.get("request-id"))
reponse = brute.parse()  # l'objet Message habituel
print(reponse.content[0].text)

Workspace : wrkspc_015TZiDsvDJZtFb9dty8A2Hu
Request ID: req_011CfMu3QF3qeFYDpguW6B1H
Ok


In [5]:
reponse = client.messages.create(
    model=MODELE,
    max_tokens=300,
    messages=[{"role": "user", "content": "Explique en deux phrases ce qu'est un RAG."}],
)


print(reponse.content[0].text)
print("-" * 60)
print(json.dumps(reponse.model_dump(), indent=2, ensure_ascii=False))

**RAG (Retrieval-Augmented Generation)** est une technique en intelligence artificielle qui combine un système de récupération d'informations avec un modèle de génération de texte, permettant à ce dernier de puiser dans une base de connaissances externe (documents, bases de données) avant de formuler sa réponse. Cette approche améliore la précision et la fiabilité des réponses générées, tout en réduisant les risques d'hallucinations, car le modèle s'appuie sur des sources concrètes plutôt que uniquement sur ses connaissances internes figées lors de l'entraînement.
------------------------------------------------------------
{
  "id": "msg_011CfMu84cjVEv2neaUEtUAR",
  "container": null,
  "content": [
    {
      "citations": null,
      "text": "**RAG (Retrieval-Augmented Generation)** est une technique en intelligence artificielle qui combine un système de récupération d'informations avec un modèle de génération de texte, permettant à ce dernier de puiser dans une base de connaissance

Petit helper pour la suite

In [17]:
def afficher(reponse: anthropic.types.Message) -> None:
    """Affiche le texte, la raison d'arrêt et la consommation de tokens."""
    texte = "".join(b.text for b in reponse.content if b.type == "text")
    print(texte)
    print(f"\n[stop_reason={reponse.stop_reason} | "
          f"in={reponse.usage.input_tokens} | out={reponse.usage.output_tokens}]")

Prompt système et température

In [20]:
SYSTEME = (
    "Tu es un assistant d'arbitrage de règles de jeu de plateau. "
    "Tu réponds en français, de façon concise et factuelle."
)
questions = {
    "créative": "Donne-moi un nom original pour une escouade de soldats futuristes.",
    "factuelle": "Combien de faces a un dé à six faces ? Réponds par un nombre.",
}


In [21]:
r = client.messages.create(
    model=MODELE, max_tokens=1000, system=SYSTEME,
    messages=[{"role": "user", "content": questions["factuelle"]}],
)
for bloc in r.content:
    print(bloc.type, "→", (getattr(bloc, "thinking", None) or getattr(bloc, "text", ""))[:200])
print(r.stop_reason, r.usage)

text → 6
end_turn Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=80, output_tokens=3, output_tokens_details=OutputTokensDetails(thinking_tokens=0), server_tool_use=None, service_tier='standard')


In [22]:
def texte_de(reponse: anthropic.types.Message) -> str:
    """Concatène uniquement les blocs texte (ignore thinking, tool_use...)."""
    return "".join(b.text for b in reponse.content if b.type == "text")

for nature, question in questions.items():
    print(f"=== question {nature} ===")
    for _ in range(3):
        r = client.messages.create(
            model=MODELE, max_tokens=1000, system=SYSTEME,
            messages=[{"role": "user", "content": question}],
        )
        print(f"- {texte_de(r).strip()}  [stop={r.stop_reason}, out={r.usage.output_tokens}]")

=== question créative ===
- **Escouade Nyx-7 "Spectres de Vérine"**

Quelques variantes si tu veux d'autres pistes :
- **Cohorte Écho-Noire**
- **Unité Fantôme Cendres**
- **Escadron Vorlex**
- **Les Chiens de Fer d'Orion**

Dis-moi le ton que tu recherches (sombre, héroïque, mercenaire, high-tech...) et je peux affiner la proposition.  [stop=end_turn, out=218]
- **Escouade Némésis-7**

D'autres suggestions dans le même esprit :
- **Spectre Sigma**
- **Faucons de Vulcain**
- **Unité Cendre-Noire**
- **Bataillon Éclipse**

Je peux affiner selon l'univers (militaire pur, cyberpunk, space opera...) si tu précises le contexte.  [stop=end_turn, out=209]
- **« La Cohorte Spectrale »**

Alternatives originales :
- **Vecteur Sinistre**
- **Escadrille Némésis-9**
- **Les Chiens de Fer**
- **Bataillon Cendrenoire**
- **Unité Éclipse-7**
- **Les Spectres du Néant**

Si tu me donnes plus de contexte (univers, ton du jeu, faction ennemie, esthétique), je peux affiner la proposition pour qu'elle col

Troncature par max_tokens

In [23]:
r = client.messages.create(
    model=MODELE, max_tokens=15,
    messages=[{"role": "user", "content": "Décris le déroulé d'un tour de jeu de figurines."}],
)
afficher(r)  # attendu : stop_reason=max_tokens

# Déroulé type d'un tour de je

[stop_reason=max_tokens | in=28 | out=15]


Multi-tours : l'API est sans état

In [ ]:
historique = [
    {"role": "user", "content": "Retiens ce nombre : 42."},
]
r1 = client.messages.create(model=MODELE, max_tokens=50, messages=historique)
historique.append({"role": "assistant", "content": r1.content})
historique.append({"role": "user", "content": "Quel nombre t'ai-je demandé de retenir ?"})

r2 = client.messages.create(model=MODELE, max_tokens=50, messages=historique)
afficher(r2)

# Contre-test : même question sans l'historique
r3 = client.messages.create(model=MODELE, max_tokens=50,
                            messages=[historique[-1]])
afficher(r3)

Compter les tokens avant d'envoyer

In [ ]:
extrait = "Texte de règle factice. " * 200
compte = client.messages.count_tokens(
    model=MODELE, system=SYSTEME,
    messages=[{"role": "user", "content": extrait}],
)
print(compte.input_tokens)

Streaming et latence

In [ ]:
debut = time.perf_counter()
premier_token = None

with client.messages.stream(
    model=MODELE, max_tokens=300,
    messages=[{"role": "user", "content": "Explique en un paragraphe ce qu'est un jet de sauvegarde."}],
) as flux:
    for morceau in flux.text_stream:
        if premier_token is None:
            premier_token = time.perf_counter() - debut
        print(morceau, end="", flush=True)
    finale = flux.get_final_message()

print(f"\n\n[1er token : {premier_token:.2f}s | total : {time.perf_counter() - debut:.2f}s]")
print(finale.usage)

Gestion des erreurs

In [ ]:
client_test = anthropic.Anthropic(max_retries=2, timeout=30.0)

try:
    client_test.messages.create(
        model="modele-inexistant", max_tokens=10,
        messages=[{"role": "user", "content": "test"}],
    )
except anthropic.NotFoundError as e:
    print("Modèle introuvable :", e.status_code)
except anthropic.RateLimitError:
    print("Rate limit (429)")
except anthropic.APIStatusError as e:
    print("Autre erreur API :", e.status_code, e.message)
except anthropic.APIConnectionError:
    print("Problème réseau")

Ce que le modèle sait déjà (pont vers le RAG)

In [ ]:
QUESTION_REGLE = (
    "En 11e édition de Warhammer 40,000, explique précisément comment se résout "
    "une attaque au tir, étape par étape. Cite la page du livre de règles."
)
r = client.messages.create(model=MODELE, max_tokens=600, temperature=0,
                           messages=[{"role": "user", "content": QUESTION_REGLE}])
afficher(r)